## Notebook 8: Dynamic CBA

**New point: The feedback of AC on temperature**
- Air conditioning systems work by extracting heat from indoor spaces and rejecting it outdoors. This waste heat warms the ambient outdoor air temperature, creating a positive feedback loop: Higher outdoor T → More AC demand → More waste heat → Even higher outdoor T.
  
- This effect is disproportionately strong at night because:
    - During daytime: The deep convective boundary layer (1–3 km) disperses waste heat
      efficiently
    - During nighttime: The shallow stable boundary layer (100–300 m) traps waste heat
      near the surface

- **Implementation**:
- adds an AC waste-heat externality term to the policy scenarios: Net AC benefits = Avoided deaths from indoor cooling − Extra deaths from outdoor warming
- switchable sensitivity
    - On/off toggle via DELTA_T_CASE = "off" | "low" | "central" | "high"
    - Configurable driver: DELTA_T_DRIVER = "kwh" | "users"
- Nighttime → Daily-mean conversion (HALVING THE EFFECT):
The literature reports nighttime warming (0.5–1.0°C), but our hazard uses daily-mean T2M.
Since daily mean ≈ (Tmax + Tmin)/2 and AC waste heat only warms Tmin (night), we must halve the literature values:
    - Literature nighttime:  low=0.50°C,  central=0.75°C,  high=1.00°C
    - Applied daily-mean:    low=0.25°C,  central=0.375°C, high=0.50°C
- Incremental approach for CBA: We compute the incremental externality (policy − baseline), not absolute warming, because both scenarios have some waste-heat impact.
- Linear scaling with AC activity: We scale warming linearly with kWh consumption (or users), capped at the literature maximum.

In [1]:
import os
os.environ["CITY"] = "rome"   # pick the city here

In [2]:
# Generic bootstrap 
from pathlib import Path
import os, sys

def _find_root():
    start = Path.cwd()
    for cand in [start, *start.parents]:
        if (cand/"cityheat").is_dir() and (cand/"configs").is_dir():
            return cand
    raise RuntimeError("Repo root not found.")
ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cityheat.nbsetup import bootstrap
from cityheat.paths import make_P, ensure_out

# Choose city here
SLUG = globals().get("SLUG", os.environ.get("CITY", "rome")).lower()

C    = bootstrap(SLUG)      # reads configs/<slug>.yml and syncs that city only if wanted
CFG  = C["CFG"]; CITY = C["CITY"]
BASE = C["BASE"]; OUT = C["OUT"]; INT = C["INT"]

P    = make_P(BASE)         # read-only path helper
OUTP = ensure_out(OUT)      # write-safe path helper
print(f"→ City: {CITY}  |  BASE={BASE}  OUT={OUT}  INT={INT}")

→ City: rome  |  BASE=/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome  OUT=/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/outputs/rome  INT=/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/outputs/rome/interim


In [3]:
# city config + file paths from YAML 
cfg = C.get("cfg", {})                      # full YAML for the selected city
SLUG = cfg.get("slug", SLUG).lower()        
CITY = cfg.get("city_name", CITY)

paths    = cfg.get("files", {})             # {gvi_csv, lcz_candidates, cooling_coeffs_csv, ...}
osm_cfg  = cfg.get("osm", {})               # OSM settings used later in NB5
trees_cfg = cfg.get("trees", {})            # TARGET/CAP for NB5
urbclim   = cfg.get("urbclim", {})          # UrbClim folder/settings for NB5

lcz_candidates = [P(p) for p in paths.get("lcz_candidates", [])]
gvi_path = P(paths.get("gvi_csv", ""))

# FUA geopackage written in NB2 
fua_gpkg = Path(paths.get("fua_gpkg", f"{OUT}/{SLUG}_fua.gpkg"))

cool_csv = P(paths.get("cooling_coeffs_csv", ""))

# checks
print("SLUG/CITY:", SLUG, CITY)
print("GVI CSV:  ", gvi_path)
print("LCZ cand: ", [str(p) for p in lcz_candidates])
print("FUA GPKG: ", fua_gpkg)
print("Cooling CSV:", cool_csv)

SLUG/CITY: rome Rome
GVI CSV:   /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome/gviRome/gvi_Rome.csv
LCZ cand:  ['/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome/LCZ/lcz_filter_v3.tif', '/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome/LCZ/lcz_v3.tif']
FUA GPKG:  /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/outputs/rome/rome_fua.gpkg
Cooling CSV: /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome/CoolingEff/outer_2_wbgt_max.csv


**Loading from before**

- Trees policy table and citywide ΔGVI diagnostics (sum of Municipi)
- AC coverage maps + population grid
- Municipality-level coverage and kWh/user outputs from earlier notebooks
- AC efficacy by age (used on benefit side)

In [4]:
# loading everything needed for the CBA
from pathlib import Path
import json
import numpy as np
import pandas as pd

OUT = Path(OUT)
INT = Path(INT)
TAB_DIR = OUT / "tables"

# Vegetation policy / ΔGVI
trees_tbl = pd.read_csv(TAB_DIR / f"{SLUG}_trees_tbl.csv")

# diagnostics JSON with citywide ΔGVI (pop-weighted, points on 1–100 scale)
veg_diag_path = OUT / f"{SLUG}_veg_diagnostics.json"
veg_diag = json.loads(veg_diag_path.read_text())
citywide_dGVI_points_popw = veg_diag["citywide_dGVI_points_popw"]
print("Citywide pop-weighted ΔGVI (points, 1–100 scale):", citywide_dGVI_points_popw)

# AC coverage / municipal pop / energy use
# coverage maps
ac_cov_npz = np.load(INT / f"ac_coverage_maps_{SLUG}.npz")
coverage_base_3d   = ac_cov_npz["coverage_base_3d"]
coverage_policy_3d = ac_cov_npz["coverage_policy_3d"]
YEARS_AC           = ac_cov_npz["YEARS_AC"]
CITY_MASK = ac_cov_npz["CITY_MASK"].astype(bool)
HGT = int(ac_cov_npz["HGT"])
WDT = int(ac_cov_npz["WDT"])

# population on ref grid
pop_npz = np.load(INT / f"pop_on_ref_{SLUG}.npz")
pop_on_ref = pop_npz["pop"]

# municipio coverage table (pop_muni, ac_base_muni, ac_policy_muni)
muni_cov = pd.read_csv(OUT / f"muni_cov_{SLUG}.csv")

# AC consumption per municipality (kWh per AC user)
muni_tbl_all = pd.read_csv(OUT / f"{SLUG}_muni_ac_consumption_summary.csv")

# AC efficacy by age (for benefits, not directly cost-side)
eff_buckets_path = INT / f"ac_eff_buckets_{SLUG}.json"
EFF_BUCKETS = json.loads(eff_buckets_path.read_text())
EFF_BUCKETS

Citywide pop-weighted ΔGVI (points, 1–100 scale): 2.3197


{'<15': 0.3, '15-64': 0.4, '65+': 0.55}

**Discount helpers**

- Set discount rate R and horizon T
- Define PV helpers and annuity factor (PV -> equivalent annual cost)
- Convention: flows are treated as end-of-year (years 1..T)

In [5]:
import numpy as np

R = 0.03 # discount rate
T = 25 # time horizon

TREE_RAMP_YEARS = 12          # maturity ramp for both benefits and O&M
TREE_START_AGE_CENTRAL = 5    # central case (pre-grown trees)
TREE_START_AGE_SENS = 0       # sensitivity 

TREE_START_AGE_YEARS = TREE_START_AGE_CENTRAL  # the one used by the model by default

# Convention: all flows occur at END of each year => years 1..T

# present value (PV) of a constant annual flow over T years.
def pv_level_flow(annual, r=R, T=T):
    """PV of a constant annual amount paid in years 1..T."""
    yrs = np.arange(1, T+1, dtype=float)
    return float(np.sum(annual * (1 + r) ** (-yrs)))

# PV of buying an item at t=0 and then replacing it every 'life' years within the horizon.
def pv_replacements(n_items, capex_per_item, r=R, T=T, life=20):
    """ PV of buying 'n_items' at t=0 and replacing every 'life' years within horizon T. """
    pv = 0.0
    t = 0
    while t <= T:
        pv += n_items * capex_per_item / ((1+r)**t if t > 0 else 1.0)
        t += life
    return float(pv)

# annuity_factor: present value of "1 euro per year" over T years. We later use this
# to convert any PV into a constant equivalent annual cost (EAC).
def annuity_factor(r=R, T=T):
    return (1 - (1 + r) ** (-T)) / r

AF = annuity_factor(R, T)
AF

17.413147691278027

In [6]:
def pv_capex_with_ramp(new_users_t, capex_per_user, life, r=R):
    """ PV of AC capex when policy coverage ramps up over time.

    new_users_t: 1D array length T, number of new policy users in each year (vs baseline).
    capex_per_user: installation cost per AC user.
    life: years between replacements.
    r: discount rate.

    For each cohort of new users in year t, we:
    - pay capex once at installation (year t+1 in our convention)
    - then pay the same capex again every life years (replacement)
    - discount each of these payments back to year 0 and sum them up.
    """
    T = len(new_users_t)
    pv = 0.0
    for t in range(T):  # t = 0..T-1 corresponds to years 1..T
        cohort = float(new_users_t[t])
        if cohort <= 0:
            continue
        pay_year = t  # installation in year t+1, then every 'life' years
        while pay_year < T:
            pv += cohort * capex_per_user / ((1 + r) ** (pay_year + 1))
            pay_year += life
    return float(pv)

**Trees: parametrisation of costs**

- Convert ΔGVI to “index points” and apply €/index-point CAPEX
- CAPEX: linear rollout over T years
- O&M: starts the year after planting and scales with maturity (ramp years)
- start_age shifts maturity (central=5, sensitivity=0)

- Compare alternative conventions:
    - A) O&M starts immediately (old)
    - B) O&M starts next year
    - C) O&M starts next year + maturity-scaled (current best practice)

In [7]:
# Parameters from rule and regreen study
CAPEX_PER_INDEX_PT = 10_000_000.0  # eur per 1 index point (1–100 scale)

# converting per-tree CAPEX and per-tree O&M into an O&M cost per GVI point.
CAPEX_PER_TREE = 210.0    # eur per tree (REGREEN median)
OM_PER_TREE_YR = 27.0     # eur per tree per year
LIFETIME_YEARS = 25       # tree benefit/O&M lifetime

# O&M per index point per year implied by tree-level numbers
OM_PER_INDEX_PT_YR = (OM_PER_TREE_YR / CAPEX_PER_TREE) * CAPEX_PER_INDEX_PT
print("O&M per index point per year (EUR):", round(OM_PER_INDEX_PT_YR, 0))

# Total change in GVI in index point (1–100 SCALE) from trees_tbl
# trees_tbl['dGVI'] is in 0–1 (fraction of max index); sum over Municipi, then ×100 => points
DELTA_INDEX = float(trees_tbl["dGVI"].clip(lower=0).sum()) * 100.0  # sum of municipio dGVI (0–1) => index points (0–100)
print("Total ΔGVI index points (1–100 scale):", round(DELTA_INDEX, 2))

# For comparison: citywide pop-weighted ΔGVI (diagnostics)
print("Pop-weighted ΔGVI points (diagnostic):", citywide_dGVI_points_popw)

# CAPEX: linear ramp over T years
# linear rollout over 25 years
# each year we add (ΔGVI / 25) points
# pay CAPEX once for that increment in that year
# investment schedule(t) = 10M€ * (ΔGVI/25) every year, discounted year by year.
def npv_capex_linear(delta_index_total, years=T, r=R, capex_per_index=CAPEX_PER_INDEX_PT):
    """
    PV of a linear ramp: we add delta_index_total/years index points each year over 'years',
    and pay capex_per_index per point.
    All flows are assumed at the end of years 1..years.
    """
    inc = delta_index_total / years  # index points added per year
    pv = 0.0
    for t in range(1, years + 1):  # t = 1..years
        capex_t = capex_per_index * inc
        pv += capex_t / ((1 + r) ** t)
    return float(pv)

# Paper-style "investment requirement" if all done at once 
TREES_CAPEX_T0 = CAPEX_PER_INDEX_PT * DELTA_INDEX

PV_trees_capex = npv_capex_linear(DELTA_INDEX, years=T, r=R, capex_per_index=CAPEX_PER_INDEX_PT)
print(f"Trees — Total CAPEX requirement (undiscounted): €{TREES_CAPEX_T0:,.0f}")
print(f"Trees — NPV CAPEX (linear ramp): €{PV_trees_capex:,.0f}")

# annuity factor
AF = annuity_factor(R, T)

# O&M that starts the year AFTER planting, and scales up with maturity (like benefits)
def cohort_rollout_maturity_factor_om(T, ramp_years, plant_share=None, lifetime=25, start_age_years=0):
    """
    factor[t] = sum_i plant_share[i] * maturity(age=t-i), where
      - planting year (age=0) has maturity forced to 0 => O&M starts at t+1
      - from age>=1, maturity is shifted by start_age_years (pre-grown trees)
    """
    if plant_share is None:
        plant_share = np.ones(T, dtype=float) / T
    max_age = min(T, lifetime)
    ages = np.arange(max_age + 1, dtype=float)  # 0..max_age
    maturity = np.minimum((ages + start_age_years) / ramp_years, 1.0)
    maturity[0] = 0.0  # no O&M in planting year
    return np.convolve(plant_share, maturity)[:T]

def npv_om_cohorts_scaled(delta_index_total, years=T, r=R, om_per_index_per_year=OM_PER_INDEX_PT_YR,
                         ramp_years=TREE_RAMP_YEARS, lifetime=LIFETIME_YEARS, start_age_years=0):
    plant_share = np.ones(years, dtype=float) / years
    factor_om = cohort_rollout_maturity_factor_om(
        T=years, ramp_years=ramp_years, plant_share=plant_share,
        lifetime=lifetime, start_age_years=start_age_years
    )
    om_stream = om_per_index_per_year * delta_index_total * factor_om
    yrs = np.arange(1, years + 1, dtype=float)
    pv = float(np.sum(om_stream * (1 + r) ** (-yrs)))
    return pv, om_stream

# calling with the new shift
PV_trees_om, om_stream_scaled = npv_om_cohorts_scaled(
    DELTA_INDEX, years=T, r=R, om_per_index_per_year=OM_PER_INDEX_PT_YR,
    ramp_years=TREE_RAMP_YEARS, lifetime=LIFETIME_YEARS,
    start_age_years=TREE_START_AGE_YEARS,
)

PV_trees_total = PV_trees_capex + PV_trees_om
EAC_capex_annuity = PV_trees_capex / AF
EAC_om_annuity = PV_trees_om / AF
EAC_total_annuity = PV_trees_total / AF

TREES_CAPEX_T0 = CAPEX_PER_INDEX_PT * DELTA_INDEX
EAC_capex_paper = TREES_CAPEX_T0 / ((1 + R)**T * T)

print(f"Trees — NPV O&M (scaled with maturity): €{PV_trees_om:,.0f}")
print(f"Trees — NPV total (CAPEX + O&M): €{PV_trees_total:,.0f}")
print(f"Trees — EAC CAPEX (annuity): €{EAC_capex_annuity:,.0f}/yr")
print(f"Trees — EAC O&M (annuity): €{EAC_om_annuity:,.0f}/yr")
print(f"Trees — EAC total (annuity): €{EAC_total_annuity:,.0f}/yr")

O&M per index point per year (EUR): 1285714.0
Total ΔGVI index points (1–100 scale): 30.23
Pop-weighted ΔGVI points (diagnostic): 2.3197
Trees — Total CAPEX requirement (undiscounted): €302,333,765
Trees — NPV CAPEX (linear ramp): €210,583,300
Trees — NPV O&M (scaled with maturity): €243,071,452
Trees — NPV total (CAPEX + O&M): €453,654,751
Trees — EAC CAPEX (annuity): €12,093,351/yr
Trees — EAC O&M (annuity): €13,959,076/yr
Trees — EAC total (annuity): €26,052,427/yr


In [8]:
print("om_stream first 5:", np.round(om_stream_scaled[:5], 2))
print("om_stream last  5:", np.round(om_stream_scaled[-5:], 2))
print("om_stream max:", float(np.max(om_stream_scaled)))
print("om_stream last:", float(om_stream_scaled[-1]))

om_stream first 5: [      0.    777429.68 1684430.97 2721003.88 3887148.4 ]
om_stream last  5: [28376183.34 29931042.7  31485902.06 33040761.42 34595620.78]
om_stream max: 34595620.77839674
om_stream last: 34595620.77839674


In [9]:
import numpy as np
import pandas as pd

def pv_from_stream(stream, r, T):
    yrs = np.arange(1, T+1, dtype=float)
    return float(np.sum(np.asarray(stream, float) * (1 + r) ** (-yrs)))

def om_stream_constant(delta_index_total, om_per_index_per_year, T, include_planting_year: bool):
    # Constant O&M per cohort (no maturity scaling)
    inc = delta_index_total / T
    stream = np.zeros(T, dtype=float)
    for t in range(1, T+1):  # cashflows in years 1..T
        active = (t if include_planting_year else max(t-1, 0))
        stream[t-1] = active * om_per_index_per_year * inc
    return stream

def om_stream_scaled(delta_index_total, om_per_index_per_year, T, ramp_years):
    # Scaled with “maturity”, and zero in planting year by construction
    plant_share = np.ones(T, dtype=float) / T
    ages = np.arange(T+1, dtype=float)  # 0..T
    maturity = np.minimum(ages / ramp_years, 1.0)  # age 0 => 0
    factor = np.convolve(plant_share, maturity)[:T]
    return om_per_index_per_year * delta_index_total * factor

TREE_OM_RAMP_YEARS = TREE_RAMP_YEARS

# A) very old convention: O&M starts immediately
sA = om_stream_constant(DELTA_INDEX, OM_PER_INDEX_PT_YR, T, include_planting_year=True)
PV_A = pv_from_stream(sA, R, T)

# B) point 1: O&M starts the year after planting
sB = om_stream_constant(DELTA_INDEX, OM_PER_INDEX_PT_YR, T, include_planting_year=False)
PV_B = pv_from_stream(sB, R, T)

# C) point 2: O&M starts next year AND scales with maturity
sC = om_stream_scaled(DELTA_INDEX, OM_PER_INDEX_PT_YR, T, ramp_years=TREE_OM_RAMP_YEARS)
PV_C = pv_from_stream(sC, R, T)

rows = []
for name, PV_om, stream in [
    ("A) O&M starts same year (old)", PV_A, sA),
    ("B) O&M starts next year", PV_B, sB),
    ("C) O&M next year + maturity-scaled", PV_C, sC),
]:
    PV_total = PV_trees_capex + PV_om
    rows.append({
        "scenario": name,
        "PV_capex": PV_trees_capex,
        "PV_om": PV_om,
        "PV_total": PV_total,
        "EAC_capex": PV_trees_capex / AF,
        "EAC_om": PV_om / AF,
        "EAC_total": PV_total / AF,
        "PV_om_over_PV_capex": PV_om / PV_trees_capex,
        "OM_year1": stream[0],
        "OM_yearT": stream[-1],
    })

compare_om = pd.DataFrame(rows)
compare_om

,scenario,PV_capex,PV_om,PV_total,EAC_capex,EAC_om,EAC_total,PV_om_over_PV_capex,OM_year1,OM_yearT
0,A) O&M starts same year (old),2.105833e+08,3.107336e+08,5.213169e+08,1.209335e+07,1.784477e+07,2.993812e+07,1.475585,1.554859e+06,3.887148e+07
1,B) O&M starts next year,2.105833e+08,2.836586e+08,4.942419e+08,1.209335e+07,1.628991e+07,2.838326e+07,1.347014,0.000000e+00,3.731662e+07
2,C) O&M next year + maturity-scaled,2.105833e+08,1.682367e+08,3.788200e+08,1.209335e+07,9.661474e+06,2.175482e+07,0.798908,0.000000e+00,2.876490e+07


- Trees O&M sensitivity: force PV(O&M) = 5 × PV(CAPEX)
   - Compute multiplier on O&M rate so the PV ratio hits the target
   - Used for “high O&M” sensitivity cases in summary tables

In [10]:
# point 3: sensitivity where PV(O&M) = 5 * PV(CAPEX) under the SAME timing convention
TARGET_OM_TO_CAPEX_PV_RATIO = 5.0  # sensitivity

# baseline under scenario C (O&M starts next year + maturity scaling)
PV_om_base, om_stream_base = npv_om_cohorts_scaled(
    DELTA_INDEX, years=T, r=R, om_per_index_per_year=OM_PER_INDEX_PT_YR,
    ramp_years=TREE_OM_RAMP_YEARS, lifetime=LIFETIME_YEARS
)

# scale factor to hit target PV ratio
k = (TARGET_OM_TO_CAPEX_PV_RATIO * PV_trees_capex) / PV_om_base
OM_PER_INDEX_PT_YR_5x = OM_PER_INDEX_PT_YR * k

PV_om_5x, om_stream_5x = npv_om_cohorts_scaled(
    DELTA_INDEX, years=T, r=R, om_per_index_per_year=OM_PER_INDEX_PT_YR_5x,
    ramp_years=TREE_OM_RAMP_YEARS, lifetime=LIFETIME_YEARS
)

print("O&M sensitivity scaling (scenario C timing):")
print(f" PV(CAPEX) = €{PV_trees_capex:,.0f}")
print(f" Base PV(O&M) = €{PV_om_base:,.0f}")
print(f" Target PV(O&M) = €{TARGET_OM_TO_CAPEX_PV_RATIO * PV_trees_capex:,.0f}")
print(f" Multiplier k on OM rate = {k:.2f}x")
print(f" New PV(O&M) = €{PV_om_5x:,.0f}")

# optional: EAC comparison
print(f" Base EAC(O&M) = €{PV_om_base/AF:,.0f}/yr")
print(f" New EAC(O&M) = €{PV_om_5x/AF:,.0f}/yr")

O&M sensitivity scaling (scenario C timing):
 PV(CAPEX) = €210,583,300
 Base PV(O&M) = €168,236,673
 Target PV(O&M) = €1,052,916,499
 Multiplier k on OM rate = 6.26x
 New PV(O&M) = €1,052,916,499
 Base EAC(O&M) = €9,661,474/yr
 New EAC(O&M) = €60,466,753/yr


**Cost AC**

- AC cost model (dynamic coverage + dynamic kWh/user)
  - Coverage ramps by municipality over time (policy - baseline)
  - New users cohorts drive CAPEX and replacement cycles
  - Ongoing costs: maintenance + electricity (kWh/user interpolated over horizon)

In [11]:
# Dynamic AC horizon years (aligned with NB7 / CLIMADA outputs)
# We only need the start year and the list of years for the CBA horizon.
ac_city_series = pd.read_csv(INT / f"ac_per_user_city_{SLUG}.csv")
ac_city_series = ac_city_series.set_index("year").sort_index()
ELEC_START_YEAR = int(ac_city_series.index.min()) # should be 2030
ELEC_YEARS = np.arange(ELEC_START_YEAR, ELEC_START_YEAR + T, dtype=int)

In [12]:
# AC COSTS: dynamic coverage + dynamic kWh/user (+ incremental waste-heat ΔT)
AC_CAPEX_PER_USER   = 500.0   # € per AC unit
AC_MAINT_RATE       = 0.05    # fraction of CAPEX per year
AC_LIFETIME_YEARS   = 10      # replacement cycle
TARIFF_EUR_PER_KWH  = 0.25    # €/kWh

maint_per_user_yr = AC_MAINT_RATE * AC_CAPEX_PER_USER

# Electricity-horizon years (keep separate from benefits YEARS later)
ELEC_YEARS = ELEC_YEARS.copy()
assert len(ELEC_YEARS) == T

# Municipio level coverage over time
try:
    muni_cov_all = pd.read_csv(OUT / f"{SLUG}_muni_cov_yearly.csv")
except FileNotFoundError:
    muni_cov_all = muni_cov.copy()

if "muni_id" in muni_cov_all.columns:
    muni_cov_all = muni_cov_all.loc[muni_cov_all["muni_id"] > 0].copy()

# Interpolate base and policy shares to full electricity horizon
rows = []
for muni_id, g in muni_cov_all.groupby("muni_id"):
    g = g.sort_values("year")
    known_years = g["year"].to_numpy(int)

    known_base = g["ac_base_muni"].to_numpy(float)
    known_pol  = g["ac_policy_muni"].to_numpy(float)
    pop_muni   = float(g["pop_muni"].iloc[0])  # assume constant

    base_t = np.interp(ELEC_YEARS, known_years, known_base)
    pol_t  = np.interp(ELEC_YEARS, known_years, known_pol)

    base_t[ELEC_YEARS <= known_years[0]] = known_base[0]
    base_t[ELEC_YEARS >= known_years[-1]] = known_base[-1]
    pol_t[ELEC_YEARS <= known_years[0]] = known_pol[0]
    pol_t[ELEC_YEARS >= known_years[-1]] = known_pol[-1]

    base_t = np.clip(base_t, 0.0, 1.0)
    pol_t  = np.clip(pol_t , 0.0, 1.0)
    dshare_t = np.clip(pol_t - base_t, 0.0, 1.0)

    for year, bs, ps, ds in zip(ELEC_YEARS, base_t, pol_t, dshare_t):
        rows.append({
            "year": int(year),
            "muni_id": muni_id,
            "pop_muni": pop_muni,
            "base_share_t": float(bs),
            "policy_share_t": float(ps),
            "dshare_t": float(ds),
        })

cov_yearly = pd.DataFrame(rows)
cov_yearly["year"] = cov_yearly["year"].astype(int)

# Users: baseline vs policy -> incremental users + cohorts 
users_base_t = (
    cov_yearly.assign(users=lambda d: d["pop_muni"] * d["base_share_t"])
    .groupby("year")["users"].sum()
    .reindex(ELEC_YEARS).to_numpy(float)
)

users_policy_t = (
    cov_yearly.assign(users=lambda d: d["pop_muni"] * d["policy_share_t"])
    .groupby("year")["users"].sum()
    .reindex(ELEC_YEARS).to_numpy(float)
)

added_users_t = users_policy_t - users_base_t  # incremental (can be negative in theory)
new_users_t = np.empty_like(added_users_t)
new_users_t[0] = added_users_t[0]
new_users_t[1:] = added_users_t[1:] - added_users_t[:-1]
new_users_t = np.maximum(new_users_t, 0.0)     # cohorts must be non-negative

added_users_final = float(added_users_t[-1])
print(f"AC — added users in final year ≈ {added_users_final:,.0f}")

# kWh/user by municipality -> interpolate to full horizon 
muni_tbl_all = pd.read_csv(OUT / f"{SLUG}_muni_ac_consumption_summary.csv")
rows_kwh = []
for muni_id, g in muni_tbl_all.groupby("muni_id"):
    g = g.sort_values("year")
    known_years = g["year"].to_numpy(int)
    vals = g["kwh_per_user_muni"].to_numpy(float)

    kwh_interp = np.interp(ELEC_YEARS, known_years, vals)
    kwh_interp[ELEC_YEARS <= known_years[0]] = vals[0]
    kwh_interp[ELEC_YEARS >= known_years[-1]] = vals[-1]

    for y, v in zip(ELEC_YEARS, kwh_interp):
        rows_kwh.append({"year": int(y), "muni_id": muni_id, "kwh_per_user_muni": float(v)})

muni_kwh_full = pd.DataFrame(rows_kwh)
cov_yearly = (
    cov_yearly.merge(muni_kwh_full, on=["year", "muni_id"], how="left")
    .fillna({"kwh_per_user_muni": 0.0})
)

# kWh totals: baseline vs policy, plus incremental for electricity COSTS 
cov_yearly["kwh_base"]   = cov_yearly["pop_muni"] * cov_yearly["base_share_t"]   * cov_yearly["kwh_per_user_muni"]
cov_yearly["kwh_policy"] = cov_yearly["pop_muni"] * cov_yearly["policy_share_t"] * cov_yearly["kwh_per_user_muni"]

kwh_base_t   = cov_yearly.groupby("year")["kwh_base"].sum().reindex(ELEC_YEARS).to_numpy(float)
kwh_policy_t = cov_yearly.groupby("year")["kwh_policy"].sum().reindex(ELEC_YEARS).to_numpy(float)
kwh_inc_t    = kwh_policy_t - kwh_base_t  # incremental kWh (can be negative in theory)

# AC WASTE-HEAT FEEDBACK CONFIGURATION

# Reference: Salamanca et al. (2014), JGR Atmospheres
# This module adds an outdoor temperature penalty for AC waste heat.
# The penalty is scaled with AC activity and converted to extra heat deaths.

# Driver selection: scale warming with electricity consumption or user count
# "kwh" is preferred as it better captures actual heat rejection
DELTA_T_DRIVER = "kwh"   # "users" or "kwh"

# Nighttime warming at 65% AC penetration (from Salamanca et al.)
# The paper shows 0.5–1.0°C for AC65% scenario in Phoenix during extreme heat
DELTA_T_NIGHT_AT65 = {"off": 0.00, "low": 0.50, "central": 0.75, "high": 1.00}
DELTA_T_CASE = "central"

# CRITICAL: HALVING CORRECTION (nighttime to daily-mean)
# The Salamanca et al. (2014) paper reports NIGHTTIME warming (0.5–1.0°C at 65%).
# However, our hazard data uses DAILY MEAN temperature (no day/night separation).

# reasoning:
#   - Daily mean ≈ (T_max + T_min) / 2
#   - AC waste heat primarily warms T_min (nighttime), NOT T_max (daytime)
#   - If T_min increases by X°C and T_max is unchanged:
#       ΔT_dailymean = (0 + X) / 2 = X / 2
#
# Therefore, we HALVE the literature nighttime values to get daily-mean equivalents:
#   - Literature nighttime: 0.5–1.0°C  to  Daily mean: 0.25–0.5°C
#
# This correction is applied via the DAILYMEAN_FROM_NIGHT multiplier below.
DAILYMEAN_FROM_NIGHT = 0.50  # Fixed: halve nighttime values 
DAILYMEAN_CASE = "central"

PEN_REF = 0.65

# COMPUTE TEMPERATURE PENALTY TIME SERIES
# We compute warming for BOTH baseline and policy scenarios (vs. hypothetical
# "NoOut" scenario with no outdoor waste heat), then take the difference.

# The CBA uses the INCREMENTAL penalty (policy − baseline), not absolute values.
city_pop = float(np.nansum(pop_on_ref[CITY_MASK]))
if city_pop <= 0:
    raise ValueError("city_pop is non-positive; check pop_on_ref and CITY_MASK")

dT_night_at65 = float(DELTA_T_NIGHT_AT65[DELTA_T_CASE])
f_dm = float(DAILYMEAN_FROM_NIGHT)

if DELTA_T_DRIVER == "users":
    pen_base_t = users_base_t / city_pop
    pen_pol_t  = users_policy_t / city_pop

    dT_night_base_t = np.clip((dT_night_at65 / PEN_REF) * pen_base_t, 0.0, dT_night_at65)
    dT_night_pol_t  = np.clip((dT_night_at65 / PEN_REF) * pen_pol_t,  0.0, dT_night_at65)

elif DELTA_T_DRIVER == "kwh":
    # kWh-based scaling 
    # Step 1: Compute reference kWh at 65% penetration
    #         kwh_ref = (65% of city pop) × (median kWh per AC user)
    with np.errstate(divide="ignore", invalid="ignore"):
        kwh_per_user_pol_t = np.where(users_policy_t > 0, kwh_policy_t / users_policy_t, np.nan)
    med = np.nanmedian(kwh_per_user_pol_t)
    kwh_per_user_ref = float(med) if np.isfinite(med) else 0.0

    kwh_ref = (PEN_REF * city_pop) * kwh_per_user_ref
    # Step 2: Compute scaling factor (°C per kWh)
    #         At kwh_ref, we get dT_night_at65 warming
    scale = 0.0 if kwh_ref <= 0 else (dT_night_at65 / kwh_ref)
    
    # Step 3: Apply scaling to actual kWh consumption
    #         Clip at dT_night_at65 to avoid extrapolating beyond literature
    dT_night_base_t = np.clip(scale * kwh_base_t,   0.0, dT_night_at65)
    dT_night_pol_t  = np.clip(scale * kwh_policy_t, 0.0, dT_night_at65)

else:
    raise ValueError("DELTA_T_DRIVER must be 'users' or 'kwh'")

# INCREMENTAL EXTERNALITY (for CBA)

# The incremental warming is the ADDITIONAL warming caused by the policy
# beyond what already exists in the baseline.

# NOTE: No clip here - can be negative if policy somehow reduced AC use
dT_night_inc_t = dT_night_pol_t - dT_night_base_t
# HALVING CORRECTION: nighttime to daily-mean
# This is where we apply the halving (f_dm = 0.50 for central case).

# Example with central/central settings:
#   dT_night = 0.75°C (from literature)
#   f_dm = 0.50 (halving factor)
#   dT_dailymean = 0.75 × 0.50 = 0.375°C (applied to hazard)
dT_dailymean_inc_t = f_dm * dT_night_inc_t

# AC waste-heat: "paper-style" (absolute vs NoOut) + incremental (policy - baseline) 

# Paper-style (absolute) warming vs "NoOut" (i.e., no outdoor waste heat)
dT_night_vs_NoOut_base   = dT_night_base_t
dT_night_vs_NoOut_policy = dT_night_pol_t

# What CBA needs (incremental, policy - baseline)
dT_night_incremental = dT_night_inc_t

# ALSO TRACK ABSOLUTE WARMING (for reporting, matches paper-style presentation)

# These show total warming vs. a hypothetical "no outdoor waste heat" scenario
# Useful for validation against Salamanca et al. figures
dT_dm_vs_NoOut_base   = f_dm * dT_night_vs_NoOut_base
dT_dm_vs_NoOut_policy = f_dm * dT_night_vs_NoOut_policy
dT_dm_incremental     = f_dm * dT_night_incremental

print(
    f"ΔT(waste-heat) incremental | driver={DELTA_T_DRIVER}, case={DELTA_T_CASE}, dailymean_case={DAILYMEAN_CASE} | "
    f"night max={dT_night_inc_t.max():.3f}°C, dailymean max={dT_dailymean_inc_t.max():.3f}°C"
)

# AC electricity + maintenance + capex PV 
# costs should use incremental energy (clip at zero so we don't create "negative costs")
elec_eur_t = np.maximum(kwh_inc_t, 0.0) * TARIFF_EUR_PER_KWH

yrs = np.arange(1, T + 1, dtype=float)

PV_ac_elec = float(np.sum(elec_eur_t * (1 + R) ** (-yrs)))
print(f"AC — PV elec (dynamic coverage & kWh/user): €{PV_ac_elec:,.0f}")

maint_eur_t = np.maximum(added_users_t, 0.0) * maint_per_user_yr
PV_ac_maint = float(np.sum(maint_eur_t * (1 + R) ** (-yrs)))
print(f"AC — PV maint €{PV_ac_maint:,.0f}")

PV_ac_capex = pv_capex_with_ramp(
    new_users_t,
    capex_per_user=AC_CAPEX_PER_USER,
    life=AC_LIFETIME_YEARS,
    r=R,
)
print(f"AC — PV capex €{PV_ac_capex:,.0f}")

PV_ac_total = PV_ac_capex + PV_ac_maint + PV_ac_elec
print(f"AC — PV total €{PV_ac_total:,.0f}")

EAC_ac_total = PV_ac_total / AF
print(f"AC — EAC total (annuity): €{EAC_ac_total:,.0f}/yr")

AC — added users in final year ≈ 199,063
ΔT(waste-heat) incremental | driver=kwh, case=central, dailymean_case=central | night max=0.035°C, dailymean max=0.017°C
AC — PV elec (dynamic coverage & kWh/user): €868,871,567
AC — PV maint €93,209,551
AC — PV capex €281,703,895
AC — PV total €1,243,785,014
AC — EAC total (annuity): €71,427,925/yr


In [13]:
print("First 5 years of elec_eur_t:", elec_eur_t[:5])
print("Last 5 years of elec_eur_t:", elec_eur_t[-5:])

First 5 years of elec_eur_t: [58227575.18681788 56972894.72012633 55718264.92995244 54463685.81629622
 53209157.37915778]
Last 5 years of elec_eur_t: [47960842.10582834 47960842.10582834 47960842.10582834 47960842.10582834
 47960842.10582834]


In [14]:
# Report BOTH: absolute (vs NoOut) and incremental (policy - baseline) 
wasteheat_df = pd.DataFrame({
    "year": ELEC_YEARS,
    "dT_night_base_vs_NoOut": dT_night_vs_NoOut_base,
    "dT_night_policy_vs_NoOut": dT_night_vs_NoOut_policy,
    "dT_night_incremental": dT_night_incremental,
    "dT_dm_base_vs_NoOut": dT_dm_vs_NoOut_base,
    "dT_dm_policy_vs_NoOut": dT_dm_vs_NoOut_policy,
    "dT_dm_incremental": dT_dm_incremental,
})

# quick summary numbers 
rep = {
    "night_max_base": float(np.max(wasteheat_df["dT_night_base_vs_NoOut"])),
    "night_max_policy": float(np.max(wasteheat_df["dT_night_policy_vs_NoOut"])),
    "night_max_incremental": float(np.max(wasteheat_df["dT_night_incremental"])),
    "dm_max_base": float(np.max(wasteheat_df["dT_dm_base_vs_NoOut"])),
    "dm_max_policy": float(np.max(wasteheat_df["dT_dm_policy_vs_NoOut"])),
    "dm_max_incremental": float(np.max(wasteheat_df["dT_dm_incremental"])),
}
print("Waste-heat ΔT (°C) maxima over horizon:", {k: round(v, 4) for k, v in rep.items()})

# optional: save for appendix / figures
(TAB_DIR / f"{SLUG}_ac_wasteheat_timeseries.csv").write_text(wasteheat_df.to_csv(index=False))
print("Saved:", TAB_DIR / f"{SLUG}_ac_wasteheat_timeseries.csv")

Waste-heat ΔT (°C) maxima over horizon: {'night_max_base': 0.75, 'night_max_policy': 0.75, 'night_max_incremental': 0.0347, 'dm_max_base': 0.375, 'dm_max_policy': 0.375, 'dm_max_incremental': 0.0174}
Saved: /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/outputs/rome/tables/rome_ac_wasteheat_timeseries.csv


**Benefits and summary**

- Health benefits time series (avoided heat deaths)
  - Read annual avoided deaths outputs (trees, AC, both, and trees-on-top)
  - Interpolate to full horizon
  - Apply tree rollout + maturity factor (dynamic) vs “full from start” (static)

In [15]:
# Benefits and summary
import numpy as np
import pandas as pd
from pathlib import Path

HORIZON_YEARS = T
DISCOUNT_RATE = R
INT = Path(INT)

USE_SCALED_BENEFITS = False

def _nb6_path(stem: str) -> Path:
    return INT / (f"{stem}_scaled_{SLUG}.csv" if USE_SCALED_BENEFITS else f"{stem}_{SLUG}.csv")

def _read_overall(stem: str) -> pd.Series:
    p = _nb6_path(stem)
    if not p.exists():
        raise FileNotFoundError(f"Missing NB6 output: {p}")
    s = pd.read_csv(p, index_col="year")["overall"]
    s.index = s.index.astype(int)
    return s.sort_index()

def pv_of_stream(cashflows, r=DISCOUNT_RATE):
    yrs = np.arange(1, len(cashflows) + 1, dtype=float)
    return float(np.sum(np.asarray(cashflows, float) * (1 + r) ** (-yrs)))

def interpolate_to_horizon(s: pd.Series, years: np.ndarray) -> np.ndarray:
    s = s.sort_index()
    known = s.index.to_numpy(int)
    vals = s.to_numpy(float)
    out = np.interp(years, known, vals)
    out[years <= known[0]] = vals[0]
    out[years >= known[-1]] = vals[-1]
    return out

avo_trees = _read_overall("annual_heat_deaths_avoided_trees_curr_AC")
avo_ac = _read_overall("annual_heat_deaths_avoided_AC_curr_AC")
avo_both = _read_overall("annual_heat_deaths_avoided_treesplusAC_curr_AC")

try:
    avo_trees_on_top = _read_overall("annual_heat_deaths_avoided_trees_on_top_AC_curr_AC")
except FileNotFoundError:
    avo_trees_on_top = (avo_both - avo_ac).rename("overall")

# these are:
# If the full trees intervention (full ΔGVI map) were already in place
# (and effectively delivering its modeled cooling),
# what deaths would it avoid in each climate year?”
print("Trees – avoided deaths per year:")
print(avo_trees, "\n")

print("AC policy – avoided deaths per year:")
print(avo_ac, "\n")

print("Both (trees+AC vs current AC) – avoided deaths per year:")
print(avo_both, "\n")

print("Trees on top of AC (NB6 file) – avoided deaths per year:")
print(avo_trees_on_top)

# Horizon construction
START_YEAR = int(min(avo_trees.index.min(), avo_ac.index.min(), avo_both.index.min(), avo_trees_on_top.index.min()))
YEARS = np.arange(START_YEAR, START_YEAR + HORIZON_YEARS, dtype=int)

# CONVERT TEMPERATURE PENALTY TO EXTRA HEAT DEATHS
# We use the marginal deaths per °C computed in NB5 (with +1°C perturbation)
# and multiply by the daily-mean temperature penalty.

# Extra deaths (year t) = marginal_deaths_per_C(t) × ΔT_dailymean(t)
m1_path = INT / f"marginal_heat_deaths_perC_currentAC_{SLUG}.csv"
# Load marginal deaths per °C (computed in NB5 by running hazard+1°C)
M1 = pd.read_csv(m1_path, index_col="year").sort_index()
M1.index = M1.index.astype(int)

# making sure we have a total column
if "total" not in M1.columns:
    M1["total"] = M1[["<15", "15-64", "65+"]].sum(axis=1)

marginal_total_perC_t = interpolate_to_horizon(M1["total"], YEARS)  # length T

# we use NB 5 (not 4, I did a mistake before taking the old baseline)
legacy = INT / f"marginal_heat_deaths_perC_{SLUG}.csv"
if legacy.exists():
    print("Legacy NB4 marginal exists but will be ignored:", legacy)

trees_full = interpolate_to_horizon(avo_trees, YEARS)
ac_full    = interpolate_to_horizon(avo_ac, YEARS)
both_full  = interpolate_to_horizon(avo_both, YEARS)
top_full   = interpolate_to_horizon(avo_trees_on_top, YEARS)

# Trees benefits: cohort rollout (25y) + maturity (12y) + optional "pre-grown" shift
def cohort_rollout_maturity_factor(T, ramp_years, plant_share=None, start_age_years=0):
    """ factor[t] = sum_i plant_share[i] * maturity(age=t-i), where:
        - maturity(age=0) is forced to 0 -> benefits start at t+1
        - maturity for age>=1 is shifted by start_age_years (pre-grown trees)
    """
    if plant_share is None:
        plant_share = np.ones(T, dtype=float) / T  # linear rollout over T years
    ages = np.arange(T + 1, dtype=float)  # 0..T (need +1 so age=0 exists explicitly)
    maturity = np.minimum((ages + start_age_years) / ramp_years, 1.0)
    maturity[0] = 0.0  # enforce: no benefits in planting year
    return np.convolve(plant_share, maturity)[:T]

trees_factor_dynamic = cohort_rollout_maturity_factor(
    T=HORIZON_YEARS, ramp_years=TREE_RAMP_YEARS, start_age_years=TREE_START_AGE_YEARS
)
trees_factor_static = np.ones(HORIZON_YEARS, dtype=float)

def compute_streams(trees_full, ac_full, both_full, top_full, trees_factor):
    trees = trees_full * trees_factor
    ac = ac_full
    top = top_full * trees_factor
    both = ac + top
    return trees, ac, top, both

trees_dyn, ac_dyn, top_dyn, both_dyn = compute_streams(
    trees_full, ac_full, both_full, top_full, trees_factor_dynamic
)
trees_sta, ac_sta, top_sta, both_sta = compute_streams(
    trees_full, ac_full, both_full, top_full, trees_factor_static
)

# AC-driven warming penalty -> extra heat deaths (annual), unit-consistent 

# We already built dT_dailymean_t in the AC-costs cell (nighttime -> daily-mean-equivalent).
# Align change in T to the same years as benefits
dT_dm_inc_y = pd.Series(dT_dm_incremental, index=ELEC_YEARS).reindex(YEARS).to_numpy(float)
dT_dm_base_y = pd.Series(dT_dm_vs_NoOut_base, index=ELEC_YEARS).reindex(YEARS).to_numpy(float)
dT_dm_pol_y  = pd.Series(dT_dm_vs_NoOut_policy, index=ELEC_YEARS).reindex(YEARS).to_numpy(float)

# exposure factor (default 1.0 = full city exposed to warming)
AC_WASTEHEAT_EXPOSURE = 1.0

# Compute incremental mortality penalty (extra deaths from policy's waste heat)
dT_eff_inc_y = AC_WASTEHEAT_EXPOSURE * np.maximum(dT_dm_inc_y, 0.0)
warming_penalty_t = marginal_total_perC_t * dT_eff_inc_y  

# paper-style reporting: absolute penalty vs "NoOut" under baseline and policy
dT_eff_base_y = AC_WASTEHEAT_EXPOSURE * np.maximum(dT_dm_base_y, 0.0)
dT_eff_pol_y  = AC_WASTEHEAT_EXPOSURE * np.maximum(dT_dm_pol_y, 0.0)

penalty_base_vs_NoOut_t  = marginal_total_perC_t * dT_eff_base_y
penalty_policy_vs_NoOut_t = marginal_total_perC_t * dT_eff_pol_y

print("Waste-heat mortality (extra heat deaths) over horizon:")
print("  baseline vs NoOut:", round(float(penalty_base_vs_NoOut_t.sum()), 3))
print("  policy   vs NoOut:", round(float(penalty_policy_vs_NoOut_t.sum()), 3))
print("  incremental (policy-baseline):", round(float(warming_penalty_t.sum()), 3))

CLIP_NET_AT_ZERO = False

# NET AC BENEFITS (subtracting the waste-heat externality)
# Trees are unaffected by AC waste heat (no penalty).
# AC-only and Both scenarios have the penalty subtracted.
def net_ac_stream(ac_stream):
    raw = ac_stream - warming_penalty_t
    return np.maximum(raw, 0.0) if CLIP_NET_AT_ZERO else raw
# trees don't cause waste heat, 
# so they keep their full benefit. Only AC benefits are reduced by the externality.

# Net streams (trees unaffected; AC and both penalized)
trees_dyn_net = trees_dyn  # No change
top_dyn_net   = top_dyn    # Reduced by waste-heat deaths
ac_dyn_net    = net_ac_stream(ac_dyn) # Trees-on-top unaffected
both_dyn_net  = top_dyn_net + ac_dyn_net # Combined

trees_sta_net = trees_sta
top_sta_net   = top_sta
ac_sta_net    = net_ac_stream(ac_sta)
both_sta_net  = top_sta_net + ac_sta_net

# Diagnostic: does AC waste-heat wipe out tree gains? 
check = pd.DataFrame({
    "year": YEARS,
    "trees_on_top": top_dyn_net,
    "trees_only": trees_dyn_net,
    "ac_wasteheat_penalty": warming_penalty_t,
})
check["penalty_over_trees_on_top"] = check["ac_wasteheat_penalty"] / check["trees_on_top"].replace(0, np.nan)
check["penalty_over_trees_only"]   = check["ac_wasteheat_penalty"] / check["trees_only"].replace(0, np.nan)

print(check.head(10))
print(check[["trees_on_top","trees_only","ac_wasteheat_penalty"]].sum())

# Debug sanity check
dbg = pd.DataFrame({
    "year": YEARS,
    "ac_gross": ac_dyn,
    "dT_dm_base_vs_NoOut": dT_dm_base_y,
    "dT_dm_policy_vs_NoOut": dT_dm_pol_y,
    "dT_dm_incremental": dT_dm_inc_y,
    "marg_deaths_perC": marginal_total_perC_t,
    "penalty_base_vs_NoOut": penalty_base_vs_NoOut_t,
    "penalty_policy_vs_NoOut": penalty_policy_vs_NoOut_t,
    "penalty_incremental": warming_penalty_t,
})

dbg["net_ac"] = ac_dyn_net
dbg["penalty_over_gross"] = dbg["penalty_incremental"] / dbg["ac_gross"].replace(0, np.nan)

print(dbg.head(8))
print("\nSums over horizon:")
print(dbg[["ac_gross", "penalty_base_vs_NoOut", "penalty_policy_vs_NoOut", "penalty_incremental", "net_ac"]].sum())

def summarize_benefits(prefix, trees, ac, top, both, r=DISCOUNT_RATE):
    pv = {
        f"{prefix}_PV_trees": pv_of_stream(trees, r),
        f"{prefix}_PV_ac": pv_of_stream(ac, r),
        f"{prefix}_PV_top": pv_of_stream(top, r),
        f"{prefix}_PV_both": pv_of_stream(both, r),
    }
    cum = {
        f"{prefix}_CUM_trees": float(np.sum(trees)),
        f"{prefix}_CUM_ac": float(np.sum(ac)),
        f"{prefix}_CUM_top": float(np.sum(top)),
        f"{prefix}_CUM_both": float(np.sum(both)),
    }
    return pv, cum

pv_dyn, cum_dyn = summarize_benefits("DYN", trees_dyn_net, ac_dyn_net, top_dyn_net, both_dyn_net)
pv_sta, cum_sta = summarize_benefits("STA", trees_sta_net, ac_sta_net, top_sta_net, both_sta_net)

print("Cumulative avoided deaths (25y, undiscounted):")
print(
    f" DYNAMIC — Trees: {cum_dyn['DYN_CUM_trees']:.2f} | AC: {cum_dyn['DYN_CUM_ac']:.2f} | "
    f"Trees on top: {cum_dyn['DYN_CUM_top']:.2f} | Both: {cum_dyn['DYN_CUM_both']:.2f}"
)
print(
    f" STATIC — Trees: {cum_sta['STA_CUM_trees']:.2f} | AC: {cum_sta['STA_CUM_ac']:.2f} | "
    f"Trees on top: {cum_sta['STA_CUM_top']:.2f} | Both: {cum_sta['STA_CUM_both']:.2f}"
)
print("\nNote: 'STATIC' assumes full policy effect from the first year of the horizon.")
print(" 'DYNAMIC' assumes gradual rollout + tree maturity, so early benefits are smaller.")

Trees – avoided deaths per year:
year
2030    7.732308
2040    7.932971
2050    8.359307
Name: overall, dtype: float64 

AC policy – avoided deaths per year:
year
2030    44.153466
2040    38.619138
2050    41.226530
Name: overall, dtype: float64 

Both (trees+AC vs current AC) – avoided deaths per year:
year
2030    50.985410
2040    45.776605
2050    48.768085
Name: overall, dtype: float64 

Trees on top of AC (NB6 file) – avoided deaths per year:
year
2030    6.831944
2040    7.157468
2050    7.541556
Name: overall, dtype: float64
Legacy NB4 marginal exists but will be ignored: /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/outputs/rome/interim/marginal_heat_deaths_perC_rome.csv
Waste-heat mortality (extra heat deaths) over horizon:
  baseline vs NoOut: 2518.848
  policy   vs NoOut: 2535.08
  incremental (policy-baseline): 16.231
   year  trees_on_top  trees_only  ac_wasteheat_penalty  \
0  2030      0.000000    0.000000              4.550040   
1  2031      0.137290  

**Outputs variables**:
- dT_night_vs_NoOut_base: Baseline nighttime warming (vs. no waste heat), in °C
- dT_night_vs_NoOut_policy: Policy nighttime warming (vs. no waste heat), in °C
- dT_night_incremental: Incremental nighttime warming (policy − baseline), in °C
- dT_dm_incremental, Incremental daily-mean warming, in °C
- warming_penalty_t, Extra deaths from waste-heat warming, deaths/year
- ac_dyn_net, Net AC benefits after subtracting penalty, deaths avoided/year

**Interpretation**:
- Both baseline and policy cause significant waste-heat mortality (2500 deaths over 25 years vs. a hypothetical city with no outdoor AC waste heat)
- The incremental impact of the AC policy is small (16 extra deaths) because:
    - Rome already has high baseline AC penetration
    - The policy adds relatively few new users
    - Both scenarios are near the literature-based cap
- The penalty does NOT wipe out AC benefits because:
  - AC avoided deaths: 1000+ over the horizon
  - Waste-heat extra deaths: ~16 (incremental)
  - Net AC benefits remain strongly positive

**Comparison AC feedback and trees benefits**

If AC policy increases outdoor temperatures via waste heat, does that warming cancel out the cooling benefit from trees? **intuition**: Trees cool the outdoor environment (ΔT negative), while AC waste heat warms it (ΔT positive). If the warming exceeds the cooling, trees would appear useless in the "both policies" scenario.

- Trees provide 0.03°C cooling (implied from 7 avoided deaths / 260 deaths per °C)
AC waste heat adds 0.017°C warming (incremental, policy minus baseline)
Net effect: Trees still cool, just slightly less than they would without AC expansion

- The waste-heat penalty erases about 20-30% of tree benefits (higher in PV terms because the penalty is front-loaded in early years when trees are still ramping up). But 70-80% of tree benefits remain.

In [16]:
import numpy as np
import pandas as pd

eps = 1e-9

# before we did: warming_penalty_t = marginal_total_perC_t * dT_eff_inc_y  
# This converts the incremental temperature increase (°C) 
# into extra deaths using the marginal deaths-per-°C from NB5. 
# The units are consistent: (deaths/°C) × (°C) = deaths.
trees_top = np.asarray(top_dyn_net, float)         # trees-on-top (dynamic rollout+maturity)
pen = np.asarray(warming_penalty_t, float)         # incremental waste-heat deaths (policy-baseline)

# A) original diagnostics
wipe_year = pen >= trees_top - eps

cum_share = pen.sum() / max(trees_top.sum(), eps)
pv_share  = pv_of_stream(pen, R) / max(pv_of_stream(trees_top, R), eps)

trees_top_net_of_feedback = trees_top - pen
cum_net = trees_top_net_of_feedback.sum()
pv_net  = pv_of_stream(trees_top_net_of_feedback, R)

print("Years where penalty >= trees-on-top:", YEARS[wipe_year])
print("Cumulative share of trees-on-top erased:", cum_share)
print("PV share of trees-on-top erased:", pv_share)
print("Cumulative net trees-on-top after feedback:", cum_net)
print("PV net trees-on-top after feedback:", pv_net)

# B) More meaningful: ignoring years when trees ~ 0 (ramp)
min_tree = 1.0  # deaths/year threshold
mask = trees_top >= min_tree

wipe_year_masked = mask & (pen >= trees_top - eps)

cum_share_masked = pen[mask].sum() / max(trees_top[mask].sum(), eps)
pv_share_masked  = pv_of_stream(pen[mask], R) / max(pv_of_stream(trees_top[mask], R), eps)

print("\n[Masked wipe-out check]")
print(f"Threshold: trees_on_top >= {min_tree} death/yr")
print("Wipe years (masked):", YEARS[wipe_year_masked])
print("Cumulative share erased (masked):", cum_share_masked)
print("PV share erased (masked):", pv_share_masked)

# C) Temperature-equivalent comparison (does warming cancel cooling?)
# Implied cooling (°C) that would produce trees-on-top avoided deaths under the same marginal slope
marg = np.asarray(marginal_total_perC_t, float)
dT_equiv_trees_top_C = np.where(marg > 0, trees_top / marg, np.nan)
# In °C terms, how does waste-heat warming compare to tree cooling? 
# A ratio > 1 would mean waste heat exceeds tree cooling. 
# result shows no such years (after masking).

# Waste-heat incremental warming (°C) already computed earlier (daily-mean-equivalent, exposure-weighted)
dT_wasteheat_inc_C = np.asarray(dT_eff_inc_y, float)

# Ratio only where tree-equivalent cooling is positive & trees are non-trivial (optional)
ratio = np.full_like(dT_equiv_trees_top_C, np.nan, dtype=float)
ratio_mask = np.isfinite(dT_equiv_trees_top_C) & (dT_equiv_trees_top_C > 0) & mask
ratio[ratio_mask] = dT_wasteheat_inc_C[ratio_mask] / dT_equiv_trees_top_C[ratio_mask]

comp = pd.DataFrame({
    "year": YEARS,
    "trees_on_top_deaths": trees_top,
    "wasteheat_penalty_deaths": pen,
    "net_trees_on_top_deaths": trees_top - pen,
    "dT_equiv_trees_top_C": dT_equiv_trees_top_C,
    "dT_wasteheat_inc_C": dT_wasteheat_inc_C,
    "T_ratio_wasteheat_over_trees": ratio,
})

finite_ratio = comp["T_ratio_wasteheat_over_trees"].to_numpy()
finite_ratio = finite_ratio[np.isfinite(finite_ratio)]

print("\n[Temperature-equivalent check]")
print("Years with ratio > 1 (wasteheat > trees cooling):",
      comp.loc[comp["T_ratio_wasteheat_over_trees"] > 1, "year"].to_numpy())
print("Median ratio (finite, masked years):",
      float(np.median(finite_ratio)) if finite_ratio.size else np.nan)

display(comp.head(12))
comp.to_csv(TAB_DIR / f"{SLUG}_trees_vs_wasteheat_comparison.csv", index=False)
print("Saved:", TAB_DIR / f"{SLUG}_trees_vs_wasteheat_comparison.csv")

Years where penalty >= trees-on-top: [2030 2031 2032 2033 2034]
Cumulative share of trees-on-top erased: 0.2104664166935571
PV share of trees-on-top erased: 0.32581815508342343
Cumulative net trees-on-top after feedback: 60.889779759452814
PV net trees-on-top after feedback: 31.028671715228146

[Masked wipe-out check]
Threshold: trees_on_top >= 1.0 death/yr
Wipe years (masked): []
Cumulative share erased (masked): 0.0010148669365556107
PV share erased (masked): 0.0014046211543528038

[Temperature-equivalent check]
Years with ratio > 1 (wasteheat > trees cooling): []
Median ratio (finite, masked years): 0.0


,year,trees_on_top_deaths,wasteheat_penalty_deaths,net_trees_on_top_deaths,dT_equiv_trees_top_C,dT_wasteheat_inc_C,T_ratio_wasteheat_over_trees
0,2030,0.000000,4.550040,-4.550040,0.000000,0.017354,NaN
1,2031,0.137290,3.810228,-3.672938,0.000523,0.014509,NaN
2,2032,0.298872,3.068050,-2.769178,0.001136,0.011665,NaN
3,2033,0.485072,2.323507,-1.838435,0.001841,0.008820,NaN
4,2034,0.696215,1.576597,-0.880382,0.002639,0.005975,NaN
5,2035,0.932627,0.827321,0.105306,0.003529,0.003131,NaN
6,2036,1.194634,0.075680,1.118954,0.004513,0.000286,0.06335
7,2037,1.482560,0.000000,1.482560,0.005592,0.000000,0.00000
8,2038,1.773091,0.000000,1.773091,0.006678,0.000000,0.00000
9,2039,2.066225,0.000000,2.066225,0.007770,0.000000,0.00000


Saved: /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/outputs/rome/tables/rome_trees_vs_wasteheat_comparison.csv


"Wipe-Out Years" Issue: artifact of tree maturity timing. In years 2030-2034, trees are barely planted (rollout just starting), so their benefit is near zero. Meanwhile, the waste-heat penalty exists immediately. 

=> AC waste heat erodes 1/3 of the PV value of trees-on-top, but doesn’t eliminate it

In [17]:
import numpy as np
import pandas as pd

eps = 1e-9

# using already-built series 
pen = np.asarray(warming_penalty_t, float)     # incremental extra deaths from waste heat (policy - baseline)
trees_only = np.asarray(trees_dyn_net, float)  # trees only (dynamic rollout+maturity)
trees_top  = np.asarray(top_dyn_net, float)    # trees on top of AC policy (dynamic rollout+maturity)

# helper
def _pv(x): 
    return pv_of_stream(np.asarray(x, float), R)

def _shares(benefit, name, min_benefit=1.0):
    benefit = np.asarray(benefit, float)
    mask = benefit >= min_benefit

    out = {
        "compare_to": name,
        "min_benefit_threshold": float(min_benefit),
        "wipe_years_unmasked": YEARS[pen >= benefit - eps].tolist(),
        "wipe_years_masked": YEARS[(mask) & (pen >= benefit - eps)].tolist(),
        "cum_benefit": float(benefit.sum()),
        "cum_penalty": float(pen.sum()),
        "cum_share_erased_unmasked": float(pen.sum() / max(benefit.sum(), eps)),
        "pv_benefit": float(_pv(benefit)),
        "pv_penalty": float(_pv(pen)),
        "pv_share_erased_unmasked": float(_pv(pen) / max(_pv(benefit), eps)),
        "cum_share_erased_masked": float(pen[mask].sum() / max(benefit[mask].sum(), eps)) if mask.any() else np.nan,
        "pv_share_erased_masked": float(_pv(pen[mask]) / max(_pv(benefit[mask]), eps)) if mask.any() else np.nan,
        "cum_net_after_penalty": float((benefit - pen).sum()),
        "pv_net_after_penalty": float(_pv(benefit - pen)),
    }
    return out

rows = [
    _shares(trees_only, "trees_only (vs current AC)"),
    _shares(trees_top,  "trees_on_top (incremental on AC policy)"),
]

summary_penalty_vs_trees = pd.DataFrame(rows)
display(summary_penalty_vs_trees)

out_csv = TAB_DIR / f"{SLUG}_penalty_vs_trees_summary.csv"
summary_penalty_vs_trees.to_csv(out_csv, index=False)
print("Saved:", out_csv)

,compare_to,min_benefit_threshold,wipe_years_unmasked,wipe_years_masked,cum_benefit,cum_penalty,cum_share_erased_unmasked,pv_benefit,pv_penalty,pv_share_erased_unmasked,cum_share_erased_masked,pv_share_erased_masked,cum_net_after_penalty,pv_net_after_penalty
0,trees_only (vs current AC),1.0,"[2030, 2031, 2032, 2033, 2034]",[],85.551610,16.231423,0.189727,51.071544,14.995516,0.293618,0.010784,0.015253,69.320187,36.076028
1,trees_on_top (incremental on AC policy),1.0,"[2030, 2031, 2032, 2033, 2034]",[],77.121203,16.231423,0.210466,46.024188,14.995516,0.325818,0.001015,0.001405,60.889780,31.028672


Saved: /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/outputs/rome/tables/rome_penalty_vs_trees_summary.csv


- Versus trees-only, the incremental waste-heat penalty is:
    - 19% of cumulative tree benefit (16.23 / 85.55)
    - 29% of PV tree benefit (14.996 / 51.072).

- Versus trees-on-top, it’s:
    - 21% of cumulative (16.23 / 77.12)
    - 33% of PV (14.996 / 46.024)

Bottom line: under current calibration, waste heat reduces the apparent benefit of trees (especially in PV terms), but does not fully null it; net remains positive.

- Baseline mortality and percentage reductions
- 
We now read the baseline heat-attributable deaths with current AC (no new policy) from CLIMADA and:
    - compute total baseline deaths per year (summing over age classes)
    - align avoided deaths for trees, AC, and both to these baseline years
    - compute percentage reductions in baseline deaths:
      * trees_pct = % reduction with trees only
      * ac_pct    = % reduction with AC only
      * both_pct  = % reduction with both policies
      * trees_on_top_pct = extra % reduction from adding trees on top of AC

In [18]:
# building a clean "tree cost pack" for base vs 5x O&M (same timing convention)
def pack_tree_costs(PV_capex, PV_om, AF, TREES_CAPEX_T0, R, T):
    PV_total = PV_capex + PV_om
    return {
        "PV_capex": PV_capex,
        "PV_om": PV_om,
        "PV_total": PV_total,
        "EAC_capex_annuity": PV_capex / AF,
        "EAC_om_annuity": PV_om / AF,
        "EAC_total_annuity": PV_total / AF,
        # paper-style: capex-only shortcut, unchanged by O&M scenario
        "EAC_capex_paper": (TREES_CAPEX_T0 / ((1 + R) ** T)) / T,
    }

trees_cost_base = pack_tree_costs(PV_trees_capex, PV_trees_om, AF, TREES_CAPEX_T0, R, T)
trees_cost_5x = pack_tree_costs(PV_trees_capex, PV_om_5x, AF, TREES_CAPEX_T0, R, T)

In [19]:
# Summary table builder that takes a tree-cost scenario explicitly
def safe_ratio(c, b):
    return float(c / b) if (b is not None and b > 1e-9) else np.inf

def build_summary(label, cost_scenario, trees_cost, CUM_trees, CUM_ac, CUM_both, CUM_top):
    return pd.DataFrame([
        {
            "Benefit_timing": label,
            "Cost_scenario": cost_scenario,
            "Policy": "Trees only (vs current AC)",
            "PV_cost_eur": trees_cost["PV_total"],
            "avoided_deaths_cum": CUM_trees,
            "Cost_per_avoided_death_eur": safe_ratio(trees_cost["PV_total"], CUM_trees),
            "EAC_capex_annuity_eur_per_yr": trees_cost["EAC_capex_annuity"],
            "EAC_om_annuity_eur_per_yr": trees_cost["EAC_om_annuity"],
            "EAC_total_annuity_eur_per_yr": trees_cost["EAC_total_annuity"],
            "EAC_capex_paper_eur_per_yr": trees_cost["EAC_capex_paper"],
            "added_AC_users": 0.0,
        },
        {
            "Benefit_timing": label,
            "Cost_scenario": cost_scenario,
            "Policy": "AC only (vs current AC)",
            "PV_cost_eur": PV_ac_total,
            "avoided_deaths_cum": CUM_ac,
            "Cost_per_avoided_death_eur": safe_ratio(PV_ac_total, CUM_ac),
            "EAC_capex_annuity_eur_per_yr": 0.0,
            "EAC_om_annuity_eur_per_yr": 0.0,
            "EAC_total_annuity_eur_per_yr": EAC_ac_total,
            "EAC_capex_paper_eur_per_yr": np.nan,
            "added_AC_users": added_users_final,
        },
        {
            "Benefit_timing": label,
            "Cost_scenario": cost_scenario,
            "Policy": "Both (trees+AC vs current AC)",
            "PV_cost_eur": trees_cost["PV_total"] + PV_ac_total,
            "avoided_deaths_cum": CUM_both,
            "Cost_per_avoided_death_eur": safe_ratio(trees_cost["PV_total"] + PV_ac_total, CUM_both),
            "EAC_capex_annuity_eur_per_yr": trees_cost["EAC_capex_annuity"],
            "EAC_om_annuity_eur_per_yr": trees_cost["EAC_om_annuity"],
            "EAC_total_annuity_eur_per_yr": trees_cost["EAC_total_annuity"] + EAC_ac_total,
            "EAC_capex_paper_eur_per_yr": trees_cost["EAC_capex_paper"],
            "added_AC_users": added_users_final,
        },
        {
            "Benefit_timing": label,
            "Cost_scenario": cost_scenario,
            "Policy": "Trees (incremental, on top of AC policy)",
            "PV_cost_eur": trees_cost["PV_total"],
            "avoided_deaths_cum": CUM_top,
            "Cost_per_avoided_death_eur": safe_ratio(trees_cost["PV_total"], CUM_top),
            "EAC_capex_annuity_eur_per_yr": trees_cost["EAC_capex_annuity"],
            "EAC_om_annuity_eur_per_yr": trees_cost["EAC_om_annuity"],
            "EAC_total_annuity_eur_per_yr": trees_cost["EAC_total_annuity"],
            "EAC_capex_paper_eur_per_yr": trees_cost["EAC_capex_paper"],
            "added_AC_users": 0.0,
        },
    ])

# building base + sensitivity summaries for BOTH benefit-timing conventions
if "trees_cost_base" not in globals():
    trees_cost_base = pack_tree_costs(PV_trees_capex, PV_trees_om, AF, TREES_CAPEX_T0, R, T)
if "trees_cost_5x" not in globals():
    trees_cost_5x = pack_tree_costs(PV_trees_capex, PV_om_5x, AF, TREES_CAPEX_T0, R, T)

summary_dynamic_base = build_summary(
    "Dynamic rollout + tree maturity", "Base O&M", trees_cost_base,
    cum_dyn["DYN_CUM_trees"], cum_dyn["DYN_CUM_ac"], cum_dyn["DYN_CUM_both"], cum_dyn["DYN_CUM_top"]
)

summary_dynamic_5x = build_summary(
    "Dynamic rollout + tree maturity", "O&M PV = 5× CAPEX PV", trees_cost_5x,
    cum_dyn["DYN_CUM_trees"], cum_dyn["DYN_CUM_ac"], cum_dyn["DYN_CUM_both"], cum_dyn["DYN_CUM_top"]
)

summary_static_base = build_summary(
    "Static (full effect from start)", "Base O&M", trees_cost_base,
    cum_sta["STA_CUM_trees"], cum_sta["STA_CUM_ac"], cum_sta["STA_CUM_both"], cum_sta["STA_CUM_top"]
)

summary_static_5x = build_summary(
    "Static (full effect from start)", "O&M PV = 5× CAPEX PV", trees_cost_5x,
    cum_sta["STA_CUM_trees"], cum_sta["STA_CUM_ac"], cum_sta["STA_CUM_both"], cum_sta["STA_CUM_top"]
)

summary_all = pd.concat(
    [summary_dynamic_base, summary_dynamic_5x, summary_static_base, summary_static_5x],
    ignore_index=True
).round(2)

summary_all

def run_tree_age_case(tree_start_age_years: int):
    # TREE COSTS (capex unchanged; O&M depends on start age)
    PV_trees_om_case, _ = npv_om_cohorts_scaled(
        DELTA_INDEX, years=T, r=R, om_per_index_per_year=OM_PER_INDEX_PT_YR,
        ramp_years=TREE_RAMP_YEARS, lifetime=LIFETIME_YEARS,
        start_age_years=tree_start_age_years,
    )
    trees_cost_case = pack_tree_costs(PV_trees_capex, PV_trees_om_case, AF, TREES_CAPEX_T0, R, T)

    # TREE BENEFITS (dynamic rollout + maturity, shifted by start age)
    trees_factor_dynamic_case = cohort_rollout_maturity_factor(
        T=HORIZON_YEARS, ramp_years=TREE_RAMP_YEARS, start_age_years=tree_start_age_years,
    )
    trees_dyn_case, ac_dyn_case, top_dyn_case, _ = compute_streams(
        trees_full, ac_full, both_full, top_full, trees_factor_dynamic_case
    )

    # NET AC (same penalty vector as main run)
    ac_dyn_case_net = net_ac_stream(ac_dyn_case)
    both_dyn_case_net = top_dyn_case + ac_dyn_case_net

    _, cum_case = summarize_benefits(
        "DYN", trees_dyn_case, ac_dyn_case_net, top_dyn_case, both_dyn_case_net
    )

    return build_summary(
        label=f"Dynamic rollout + maturity (start_age={tree_start_age_years})",
        cost_scenario="Base O&M",
        trees_cost=trees_cost_case,
        CUM_trees=cum_case["DYN_CUM_trees"],
        CUM_ac=cum_case["DYN_CUM_ac"],
        CUM_both=cum_case["DYN_CUM_both"],
        CUM_top=cum_case["DYN_CUM_top"],
    )

summary_central = run_tree_age_case(5)
summary_sens0 = run_tree_age_case(0)
summary_tree_age = pd.concat([summary_central, summary_sens0], ignore_index=True).round(2)
summary_tree_age

,Benefit_timing,Cost_scenario,Policy,PV_cost_eur,avoided_deaths_cum,Cost_per_avoided_death_eur,EAC_capex_annuity_eur_per_yr,EAC_om_annuity_eur_per_yr,EAC_total_annuity_eur_per_yr,EAC_capex_paper_eur_per_yr,added_AC_users
0,Dynamic rollout + maturity (start_age=5),Base O&M,Trees only (vs current AC),4.536548e+08,85.55,5302702.69,12093350.58,13959075.97,26052426.56,5775851.59,0.00
1,Dynamic rollout + maturity (start_age=5),Base O&M,AC only (vs current AC),1.243785e+09,1004.46,1238267.23,0.00,0.00,71427925.36,NaN,199063.06
2,Dynamic rollout + maturity (start_age=5),Base O&M,Both (trees+AC vs current AC),1.697440e+09,1081.58,1569411.49,12093350.58,13959075.97,97480351.92,5775851.59,199063.06
3,Dynamic rollout + maturity (start_age=5),Base O&M,"Trees (incremental, on top of AC policy)",4.536548e+08,77.12,5882360.93,12093350.58,13959075.97,26052426.56,5775851.59,0.00
4,Dynamic rollout + maturity (start_age=0),Base O&M,Trees only (vs current AC),3.788200e+08,61.33,6176523.53,12093350.58,9661473.99,21754824.57,5775851.59,0.00
5,Dynamic rollout + maturity (start_age=0),Base O&M,AC only (vs current AC),1.243785e+09,1004.46,1238267.23,0.00,0.00,71427925.36,NaN,199063.06
6,Dynamic rollout + maturity (start_age=0),Base O&M,Both (trees+AC vs current AC),1.622605e+09,1059.77,1531097.72,12093350.58,9661473.99,93182749.93,5775851.59,199063.06
7,Dynamic rollout + maturity (start_age=0),Base O&M,"Trees (incremental, on top of AC policy)",3.788200e+08,55.31,6849064.13,12093350.58,9661473.99,21754824.57,5775851.59,0.00


- Baseline mortality + percent reductions
  - Read baseline heat-attributable deaths under current AC (no new policy)
  - Convert avoided deaths to % reduction for trees, AC, both, and trees-on-top

In [20]:
# Baseline mortality + percent reductions (NET, aligned to YEARS)

from pathlib import Path
import pandas as pd
import numpy as np

# Load baseline deaths under current AC (no new policy) 
candidates = [
    INT / f"annual_heat_deaths_baseline_currentAC_{SLUG}.csv",
    INT / f"annual_heat_deaths_currentAC_{SLUG}.csv",
    INT / f"baseline_heat_deaths_currentAC_{SLUG}.csv",
    INT / f"heat_deaths_baseline_currentAC_{SLUG}.csv",
]

baseline_path = next((p for p in candidates if Path(p).exists()), None)
if baseline_path is None:
    raise FileNotFoundError(
        "Could not find baseline deaths file in INT. Tried:\n" + "\n".join(map(str, candidates))
    )

baseline_df = pd.read_csv(baseline_path, index_col="year").sort_index()
baseline_df.index = baseline_df.index.astype(int)

# Ensure a single baseline_total column
if "baseline_total" not in baseline_df.columns:
    if "total" in baseline_df.columns:
        baseline_df["baseline_total"] = baseline_df["total"]
    else:
        age_cols = [c for c in ["<15", "15-64", "65+"] if c in baseline_df.columns]
        if not age_cols:
            raise ValueError(f"Baseline file columns not recognized: {list(baseline_df.columns)}")
        baseline_df["baseline_total"] = baseline_df[age_cols].sum(axis=1)

# Align baseline to our horizon YEARS
baseline_total = interpolate_to_horizon(baseline_df["baseline_total"], YEARS)

print("Baseline loaded from:", baseline_path)
print("Baseline total (first 5):", np.round(baseline_total[:5], 2))

# Convert avoided deaths into % reduction vs baseline 
def make_benefit_pct_table(label: str, baseline_total, trees, ac, both, top) -> pd.DataFrame:
    out = pd.DataFrame({
        "year": YEARS,
        "baseline_total": baseline_total,
        "avo_trees": trees,
        "avo_ac": ac,
        "avo_both": both,
        "avo_trees_on_top": top,
    }).set_index("year")

    out["trees_pct"] = 100 * out["avo_trees"] / out["baseline_total"]
    out["ac_pct"]    = 100 * out["avo_ac"]    / out["baseline_total"]
    out["both_pct"]  = 100 * out["avo_both"]  / out["baseline_total"]

    # incremental trees on top of AC, in percentage points
    out["trees_on_top_pct"] = out["both_pct"] - out["ac_pct"]

    out.insert(0, "Benefit_timing", label)
    return out

benefit_pct_dynamic = make_benefit_pct_table(
    "Dynamic rollout + tree maturity (NET)",
    baseline_total, trees_dyn_net, ac_dyn_net, both_dyn_net, top_dyn_net
).round(2)

benefit_pct_static = make_benefit_pct_table(
    "Static (full effect from start) (NET)",
    baseline_total, trees_sta_net, ac_sta_net, both_sta_net, top_sta_net
).round(2)

benefit_pct = pd.concat([benefit_pct_dynamic, benefit_pct_static])
benefit_pct

Baseline loaded from: /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/outputs/rome/interim/annual_heat_deaths_baseline_currentAC_rome.csv
Baseline total (first 5): [559.26 562.54 565.82 569.1  572.38]


,Benefit_timing,baseline_total,avo_trees,avo_ac,avo_both,avo_trees_on_top,trees_pct,ac_pct,both_pct,trees_on_top_pct
year,,,,,,,,,,
2030,Dynamic rollout + tree maturity (NET),559.26,0.00,39.60,39.60,0.00,0.00,7.08,7.08,0.00
2031,Dynamic rollout + tree maturity (NET),562.54,0.16,39.79,39.93,0.14,0.03,7.07,7.10,0.02
2032,Dynamic rollout + tree maturity (NET),565.82,0.34,39.98,40.28,0.30,0.06,7.07,7.12,0.05
2033,Dynamic rollout + tree maturity (NET),569.10,0.55,40.17,40.65,0.49,0.10,7.06,7.14,0.09
2034,Dynamic rollout + tree maturity (NET),572.38,0.78,40.36,41.06,0.70,0.14,7.05,7.17,0.12
2035,Dynamic rollout + tree maturity (NET),575.66,1.04,40.56,41.49,0.93,0.18,7.05,7.21,0.16
2036,Dynamic rollout + tree maturity (NET),578.94,1.33,40.76,41.95,1.19,0.23,7.04,7.25,0.21
2037,Dynamic rollout + tree maturity (NET),582.22,1.65,40.28,41.76,1.48,0.28,6.92,7.17,0.25
2038,Dynamic rollout + tree maturity (NET),585.49,1.97,39.73,41.50,1.77,0.34,6.79,7.09,0.30


In [21]:
assert np.isfinite(PV_trees_total) and PV_trees_total > 0
assert np.isfinite(PV_ac_total) and PV_ac_total > 0
assert abs((penalty_base_vs_NoOut_t.sum() + warming_penalty_t.sum()) - penalty_policy_vs_NoOut_t.sum()) < 1e-6

**On a PV budget**

**Sensitivity**